In modern backend systems, whether you are streaming events from Kafka or querying massive tables in PostgreSQL, loading huge datasets into memory all at once will trigger Out-Of-Memory (OOM) errors. Python solves this elegantly using the Iterator Protocol and Generators.

## 1. The Iterator Protocol (Under the Hood)
To understand generators, you first need to distinguish between two concepts that sound similar but are mechanically different: Iterables and Iterators.

* Iterable: Any object you can loop over (lists, strings, dicts). It implements the __iter__() magic method, which returns an Iterator.

* Iterator: The object that actually does the work of looping. It has state (it remembers where it is) and implements the __next__() magic method. When exhausted, it raises a StopIteration exception.

When you write a standard for loop in Python, the interpreter is actually doing this behind the scenes:

In [12]:
data = [1, 2, 3] # This is an Iterable

# Step 1: Get the iterator object from the iterable
iterator = iter(data) # Calls data.__iter__()

# Step 2: Continuously call next() until exhausted
while True:
    try:
        item = next(iterator) # Calls iterator.__next__()
        print(item)
    except StopIteration:
        # Step 3: Catch the exception to break the loop safely
        break

1
2
3


You can build your own Iterator by writing a class that implements both methods. This is exactly what a custom stream consumer does under the hood:

In [13]:
class Counter:
    def __init__(self, low, high):
        self.current = low
        self.high = high

    def __iter__(self):
        # An iterator must return itself as the iterable
        return self

    def __next__(self):
        if self.current > self.high:
            raise StopIteration
        
        # Save the current state, increment, and return
        value = self.current
        self.current += 1
        return value

for num in Counter(1, 3):
    print(num)

1
2
3


## 2. Generators: The Elegant Iterator
Writing custom classes with __iter__ and __next__ requires a lot of boilerplate. Generators are just standard Python functions that automatically implement the Iterator Protocol for you.

What makes a function a generator? A single keyword: yield.

* return vs yield
return destroys state. When a function hits a return statement, it hands back a value, tears down its local namespace (all its variables are destroyed), and terminates completely.

* yield suspends state. When a function hits yield, it hands back a value, but pauses its execution. The function's local variables, instruction pointer, and state are saved in memory. When you call next() on it again, it resumes executing exactly on the line following the yield.

In [14]:
def simple_generator():
    print("Execution starts")
    yield 1  # Pauses here on the first next()
    
    print("Execution resumes")
    yield 2  # Pauses here on the second next()
    
    print("Execution finishes")
    # Implied StopIteration is raised here

# Calling the function doesn't execute it! It returns a generator object.
gen = simple_generator() 

print(next(gen)) 
# Output: Execution starts
# Output: 1

print(next(gen))
# Output: Execution resumes
# Output: 2

Execution starts
1
Execution resumes
2


## 3. Why This Matters: Memory Efficiency
If you need to process a sequence of 10 million numbers (or API records, or log lines), generating them all at once consumes massive amounts of RAM. Generators calculate values lazily (one at a time, strictly on demand).

Notice the syntax difference between a List Comprehension (uses brackets) and a Generator Expression (uses parentheses):

In [15]:
import sys

# List Comprehension: Generates all 1,000,000 integers in memory at once
list_comp = [x * 2 for x in range(1_000_000)]
print(f"List size: {sys.getsizeof(list_comp)} bytes") 
# Output: ~8,000,000 bytes (8 MB)

# Generator Expression: Stores ONLY the formula and current state
gen_exp = (x * 2 for x in range(1_000_000))
print(f"Generator size: {sys.getsizeof(gen_exp)} bytes") 
# Output: ~104 bytes (Virtually nothing)

List size: 8448728 bytes
Generator size: 200 bytes
